# 第四课｜电路怎样“记住”上一时刻的膜电位？

前三课都还在软件世界里：Python 变量 `v` 很自然地保存了上一时刻的膜电位状态。

但我们的终点是 **现场可编程门阵列（Field-Programmable Gate Array, FPGA）**。FPGA 里面不是一个个 Python 变量，而是数字电路。

所以今天只解决一个问题：

> **数字电路怎样保存状态，并让状态在合适的时刻更新？**

本课的主要新概念：**数字状态（digital state）和时钟（clock）**。

今天仍然不要求会写 SystemVerilog。

## 1. 先认识“数字电路”是什么意思

**数字电路（digital circuit）** 使用离散的数字状态来表示和处理信息。最基础的状态通常用 **二进制位（binary digit, bit）** 表示，也就是 `0` 或 `1`。

上一课我们已经看到，多个 bit 组合起来可以表示整数和定点数。

今天我们不研究晶体管内部物理，而只关心更高一层的问题：

- 有些电路只根据“现在的输入”计算输出；
- 有些电路还必须记得“过去发生过什么”。

神经元显然属于第二种，因为下一时刻的膜电位依赖上一时刻的膜电位。

## 2. 没有记忆的电路：combinational logic

**组合逻辑（combinational logic）** 的输出只由当前输入决定。

最简单的例子就是加法：给定 `a` 和 `b`，输出 `a+b`。如果输入改变，输出也跟着改变；电路本身不负责保存昨天、上一秒或上一时钟周期的结果。

下面用 Python 函数模拟这种“只看当前输入”的关系。

In [1]:
def combinational_adder(a, b):
    return a + b

for a, b in [(1, 2), (4, 5), (10, -3)]:
    print(a, '+', b, '=', combinational_adder(a, b))

1 + 2 = 3
4 + 5 = 9
10 + -3 = 7


## 3. 为什么神经元不能只有组合逻辑？

如果 LIF 神经元只有当前输入 `I[t]`，却完全不知道上一时刻的 `V[t]`，它就无法完成 integrate。

所以我们需要某种硬件结构，把一个数保存下来，等下一次更新时继续使用。

这种小型状态存储结构通常叫 **寄存器（register）**。

你现在可以先把 register 理解成：

> **电路里一个能保存数字状态的位置。**

真实寄存器由更底层的数字存储单元构成，但今天不需要深入到晶体管或触发器结构。

## 4. 状态什么时候更新？认识 clock

如果很多寄存器都可以在任意时刻随意改变，一个复杂系统会很难协调。

同步数字系统常用一个周期性信号来约定“什么时候允许状态一起更新”。这个信号叫：

**时钟（clock）**。

时钟在高低电平之间周期变化。它发生跳变的瞬间叫 **时钟边沿（clock edge）**。很多寄存器被设计成只在某一种边沿——比如从 0 变成 1 的上升沿——接受新值。

所以可以建立一个非常重要的直觉：

> **边沿之间：电路计算 next state；边沿到来：register 保存 next state，成为新的 state。**

## 5. 用 Python 模拟“只在 clock edge 更新”

Python 本身当然不是硬件，但我们可以先用它模拟规则。

下面把每一次循环想象成一个 clock edge。`before` 是边沿到来前 register 保存的旧状态，`after_clock` 是边沿以后真正保存的新状态。

In [2]:
state = 0
next_values = [3, 7, 2, 9]

for cycle, next_value in enumerate(next_values):
    before = state

    # Imagine a clock edge here:
    state = next_value

    print(f'cycle={cycle}: before={before}, after_clock={state}')

cycle=0: before=0, after_clock=3
cycle=1: before=3, after_clock=7
cycle=2: before=7, after_clock=2
cycle=3: before=2, after_clock=9


## 6. state 和 next_state 不是同一个概念

这是学习硬件最关键的思想之一。

假设 register 当前保存：

`state = 3`

组合逻辑根据当前输入计算：

`next_state = 5`

在时钟边沿到来**之前**，register 里保存的仍然是 `3`。`5` 只是“如果下一个边沿到来，我们准备保存什么”。

边沿到来以后：

`state ← next_state`

于是新的 state 才变成 `5`。

这和上一课 LIF 里的 `candidate_v` / `stored_v` 已经开始发生呼应。

## 7. 一个最小 accumulator：已经很像神经元了

**累加器（accumulator）** 是一种不断把新输入加到已保存状态上的结构。

这正好对应 LIF 的 integrate 直觉。

我们再加一个**比较器（comparator）**：它只回答类似 `next_state >= threshold ?` 这样的比较问题。

comparator 是辅助术语，不是今天的主要学习目标；你只要知道它负责比较两个数字。

In [3]:
state = 0
threshold = 4
inputs = [1, 1, 1, 1, 2, 2]

for cycle, x in enumerate(inputs):
    before = state

    # Combinational calculation between clock edges
    next_state = state + x
    spike = next_state >= threshold
    value_to_store = 0 if spike else next_state

    # Imagine the clock edge here
    state = value_to_store

    print(
        f'cycle={cycle}: state_before={before}, input={x}, '
        f'next_state={next_state}, spike={spike}, state_after={state}'
    )

cycle=0: state_before=0, input=1, next_state=1, spike=False, state_after=1
cycle=1: state_before=1, input=1, next_state=2, spike=False, state_after=2
cycle=2: state_before=2, input=1, next_state=3, spike=False, state_after=3
cycle=3: state_before=3, input=1, next_state=4, spike=True, state_after=0
cycle=4: state_before=0, input=2, next_state=2, spike=False, state_after=2
cycle=5: state_before=2, input=2, next_state=4, spike=True, state_after=0


## 8. 现在重新读一次这段系统

把上面的代码暂时忘掉 Python 语法，只看硬件角色：

1. `state`：register 当前保存的状态；
2. `state + x`：组合逻辑计算 next state；
3. `next_state >= threshold`：comparator 判断是否达到阈值；
4. `value_to_store`：决定下一次真正要保存什么；
5. clock edge：把 `value_to_store` 写进 register。

一个非常粗略的概念图是：

```mermaid
flowchart LR
    REG["当前状态 current state<br/>(寄存器 register)"] --> CALC["组合计算 calculation"]
    CALC --> CAND["下一状态候选 next state candidate"]
    CAND --> STORE["选出待保存的值 value chosen for storage"]
    STORE --> REG
    CLK["时钟边沿 clock edge"] -.->|决定何时更新存入| REG
```

这已经非常接近未来的数字 LIF 神经元。

## 9. combinational logic 和 sequential logic 的区别

现在可以正式介绍第二个术语：

**时序逻辑（sequential logic）** 是行为依赖过去保存状态的数字逻辑。

可以粗略对比：

| 类型 | 是否需要记忆过去 | 在我们的神经元里可能做什么 |
|---|---|---|
| combinational logic | 否 | 加法、衰减计算、threshold 比较 |
| sequential logic | 是 | 保存 membrane state、refractory counter |

不要把 `sequential` 理解成“像 Python 一样一行一行执行”。这里的意思是：系统状态会随一系列 clock cycles 演化。

## 10. Try It：先画时序表，再运行

给定：

- 初始 `state = 0`
- inputs = `[2, 1, 3]`
- threshold = `5`

不要先运行。手工填写：

| cycle | state before | input | next state | spike? | state after clock |
|---|---:|---:|---:|---|---:|
| 0 | ? | 2 | ? | ? | ? |
| 1 | ? | 1 | ? | ? | ? |
| 2 | ? | 3 | ? | ? | ? |

然后修改代码验证。

如果预测和程序不同，先找自己的 state/next_state 理解问题，而不是马上让 AI 改代码。

## 11. 今天不学什么？

下面这些词以后会学，但今天只预告：

- **硬件描述语言（Hardware Description Language, HDL）**：用来描述数字硬件的语言类别；
- **SystemVerilog**：本项目以后会使用的一种硬件描述与验证语言；
- **寄存器传输级（Register-Transfer Level, RTL）**：描述寄存器保存什么、寄存器之间每个时钟周期如何计算和传递数据的设计层次。

你今天不需要会写 HDL，也不需要会写 RTL。今天只需要把 state、next state、register 和 clock edge 想清楚。

## 12. AI Task
,
,让 AI 用一张 Mermaid 图解释：
,
,`register → calculation → next_state → register`
,
,并要求它回答：
,
,- 哪些部分是 combinational logic？
,- 哪个部分保存 state？
,- clock edge 在哪里起作用？
,
,暂时不要让 AI 生成完整 LIF SystemVerilog。我们还没有学到那里。

## 13. Human Check

不用 AI，你应该能回答：

- combinational logic 为什么不够实现 LIF？
- register 是什么？
- clock 和 clock edge 分别是什么？
- state 与 next_state 为什么不能混为一谈？
- sequential logic 中的 sequential 为什么不等于“程序逐行执行”？
- 为什么我们故意还没有开始写 SystemVerilog？

## 14. Engineering Handoff

下一阶段会把今天的直觉变成真正的数字硬件微实验：

1. combinational adder；
2. clocked counter；
3. accumulator + threshold。

完成这些以后，才进入正式 LIF RTL。

这符合我们的 Learning Independence Axiom：不要第一次学 clock 的同一天，又同时要求掌握 SystemVerilog 语法和完整神经元硬件。

## 15. 项目追踪 Project Trace

- Lesson ID: `LSN-004`
- Engineering slice: `RMD-003A`
- Learning-process requirement/design: `PFR1 / PDP1`
- Prepares for product design: `FR2 / DP2`

这些 ID 是维护项目用的，不要求学习者背。

## 16. Exit Ticket

进入下一课前，你应该能够：

1. 展开并解释 FPGA、bit、register、clock、clock edge；
2. 解释 combinational logic 与 sequential logic 的区别；
3. 给一个简单 accumulator 手工写出三四个 cycle 的 state/next_state；
4. 明确今天只是建立数字状态直觉，还没有正式学习 HDL / RTL / SystemVerilog。

下一课我们会继续非常小的一步：

> 一个数字电路除了“记住状态”，还需要哪些最基本的逻辑积木？